In [1]:
from dataclasses import dataclass
import dask
import dask.dataframe as dd
from dask.distributed import Client, progress
from rich.console import Console
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import sys
sys.path.append('/home/fnoble/projects/AbstractForecast/config')
from config import config

console = Console(highlight=False)
client = Client(n_workers=4, threads_per_worker=8)

In [2]:
def pshape(df):
    print(f'{df.shape[0]:,} rows')

In [3]:
data_path = config.data.staged / 'psychology'
if data_path.exists():
    console.print(f'[green] Found data path [/green]')
else:
    console.print(f'[red] Failed to find data path [/red]')

 Found data path 


In [4]:
df = dd.read_parquet(data_path, engine='fastparquet')
df = df.persist()
progress(df)
df = df.compute()
pshape(df)

[                                        ] | 0% Completed |  0.0s

[                                        ] | 0% Completed |  0.1s

[                                        ] | 0% Completed |  0.2s

[                                        ] | 0% Completed |  0.3s

[                                        ] | 0% Completed |  0.4s

...

[#######################                 ] | 59% Completed |  1.2s

[##########################              ] | 67% Completed |  1.3s

[#####################################   ] | 93% Completed |  1.4s

[########################################] | 100% Completed |  1.5s

2,064,088 rows


In [6]:
df = df[df['language'] == 'en']
df = df[df['type'] == 'article']
pshape(df)

1,440,797 rows


In [7]:
df['abstract_len'] = df['abstract'].apply(lambda x : len(x))
plt.figure()
plt.hist(df['abstract_len'])
plt.show()
stats = df['abstract_len'].describe()
print(stats)

count    1.440797e+06
mean     1.119304e+03
std      9.922367e+02
min      1.000000e+00
25%      6.460000e+02
50%      9.890000e+02
75%      1.385000e+03
max      4.301600e+04


Name: abstract_len, dtype: float64


In [8]:
# Filter out long/short abstracts (+- 3 std)
mean = stats['mean']
std = stats['std']
limit = std * 3
high = mean + limit
low = max(mean - limit, 300)
df = df[(df['abstract_len'] > low) & (df['abstract_len'] < high)]
pshape(df)

1,285,394 rows


In [9]:
# Filter abstracts contianing 'exclude' strings, remove elements of strings fitting 'remove' regex patterns 
@dataclass(frozen=True)
class Filter:
    exclude = ['keywords:', 'Keywords:' 'query=', 'http', 'Abstract', 'Abstract '
    'ADVERTISEMENT RETURN TO ISSUE', 'PAPER ACCEPTED FOR PUBLICATION'
    'Article Views', 'Altimetric-Citations', 
    'Copyright', 'copyright', '©'
    'reference to this paper', 'Google Scholar'
    # foreign connectives (remember to include space)
    'de ',
    
    # chinese characters
   # Particles and function words
    '的', '了', '在', '是', '和', '与', '及', '或', '为', '被',
    '有', '无', '以', '对', '对于', '根据', '按', '由',
    
    # Common academic connectives
    '因此', '而且', '然而', '但', '但是', '同时', '并', '并且',
    '此外', '另外', '进一步', '总之', '综上', '可见',
    
    # Common verbs (often semantically weak in abstracts)
    '表明', '显示', '证明', '说明', '指出', '认为', '发现',
    '提出', '方法', '研究', '分析', '讨论', '介绍', 
     # Common single char words
    '的','了','是','在','和','与','对','有','无','以','为','被','等',
        # Vowels with accents
    'à', 'á', 'â', 'ã', 'ä', 'å', 'æ',
    'è', 'é', 'ê', 'ë',
    'ì', 'í', 'î', 'ï',
    'ò', 'ó', 'ô', 'õ', 'ö', 'ø', 'œ',
    'ù', 'ú', 'û', 'ü',
    'ý', 'ÿ',
    
    # Uppercase versions
    'À', 'Á', 'Â', 'Ã', 'Ä', 'Å', 'Æ',
    'È', 'É', 'Ê', 'Ë',
    'Ì', 'Í', 'Î', 'Ï',
    'Ò', 'Ó', 'Ô', 'Õ', 'Ö', 'Ø', 'Œ',
    'Ù', 'Ú', 'Û', 'Ü',
    'Ý',
    
    # Consonants with diacriticals
    'ç', 'Ç',
    'ñ', 'Ñ',
    'ð', 'Ð',
    'þ', 'Þ',
    'ß',

    ]
    tails = [
    'English', 'english',
    '<' , '>', ';', '@', '?', '[', ']', '{', '}'
    '#', '~', '/', '-', '_', '+', '=', '\\', '`', '¬', 
    '!', '£', '$', '%','^', '&', '*', '(', ')' 
    ]
    remove = []
@dataclass(frozen=True)
class Requirements:
    end_with = '.'

pshape(df)
filt = Filter()
df = df[~df['abstract'].str.contains('|'.join(filt.exclude))]
df = df[~df['abstract'].str.startswith('|'.join(filt.tails))]
#df = df[~df['abstract'].str.endswith('|'.join(filt.tails))]

req = Requirements()
df = df[df['abstract'].str.endswith(req.end_with)]
pshape(df)

1,285,394 rows


1,030,975 rows


In [23]:
def sample(n, df, cols):
    mask = np.random.randint(0,df.shape[0]-1, (n,))
    samp = df.iloc[mask, :]
    return samp[cols].reset_index()
cols = ['title', 'abstract', 'abstract_len', 'cited_by_count', 'language']
samp = sample(2, df, cols)
for i in range(samp.shape[0]):
    print('\n======') 
    for c in cols:
        print(samp.loc[i, c])

DIAZEPAM AND ITS HYDROXYLATED METABOLITES: STUDIES ON SLEEP IN HEALTHY MAN


The effects of diazepam 5, 10 and 15 mg and its hydroxylated metabolites, 3‐hydroxydiazepam (temazepam) 10, 20 and 30 mg and 3‐hydroxy, N‐desmethyldiazepam (oxazepam) 15, 30 and 45 mg on sleep in healthy man were studied in young adulthood and in middle age. The effectiveness of the drugs for sleep during the day was also investigated. In young adults diazepam and temazepam reduced sleep onset latencies and awake activity and increased total sleep time, and temazepam also reduced drowsy sleep. The activity of oxazepam was similar to that of temazepam except that it had no effect on sleep onset latencies. In middle age the effects of diazepam and temazepam were less pronounced than would be expected from studies in young adulthood. Essentially they reduced awake activity. During the day diazepam increased total sleep time and reduced drowsy sleep in young adults, but temazepam and oxazepam had less activity than would be expected from their effect on night‐time sleep. With temazepam the

1425


8


en


PSYCHOLOGICAL PARADIGMS USED IN TEACHING ANALYSIS


In this article, teaching as a subject of instruction, will be examined in terms of some actual psychological paradigms of communication: structural-expressive, formal-transactional, relational-systemic, and phenomenal-praxis. Reporting these paradigms, thispaper highlights the importance of psychological processes involved in teaching process. Related to various paradigms described, the author demonstrates the importance of addressing teaching from a high level participation in the elaboration of a mutual action.


519


0


en


In [11]:
print('starting')
df_out = dd.from_pandas(df, npartitions=64)
print('ddf converted')
df_out = dd.to_parquet(
    df_out,
    str(config.data.staged / 'psychology_clean'),
    engine = 'pyarrow',
    compression = 'zstd',
    compression_level = 1,
    write_statistics = True,
    compute = False,
    overwrite = True,
)
progress(client.compute(df_out))
print('done')

[                                        ] | 0% Completed |  8.4s

[                                        ] | 0% Completed |  8.5s

[                                        ] | 0% Completed |  8.6s

[                                        ] | 0% Completed |  8.7s

[                                        ] | 0% Completed |  8.8s

...

[#################################       ] | 83% Completed |  9.8s

[####################################    ] | 91% Completed | 10.1s

[####################################### ] | 97% Completed | 10.2s

[########################################] | 100% Completed | 10.3s

done
